In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import numpy as np
import pandas as pd
import os
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

CHECKPOINT_PATH = "/content/drive/MyDrive/greenmlops/models/distilbert_baseline.pt"
AG_NEWS_TRAIN   = "/content/drive/MyDrive/greenmlops/airflow/include/data/processed/ag_news/train.csv"
OUTPUT_DIR      = "/content/drive/MyDrive/greenmlops/airflow/include/data/embeddings/ag_news"
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df = pd.read_csv(AG_NEWS_TRAIN)
print(f"device: {DEVICE}")
print(f"AG News train shape: {df.shape}")
print(f"columns: {df.columns.tolist()}")
print(f"label distribution:\n{df['label'].value_counts().sort_index()}")

Mounted at /content/drive
device: cuda
AG News train shape: (102000, 2)
columns: ['text', 'label']
label distribution:
label
0    25691
1    25349
2    25281
3    25679
Name: count, dtype: int64


In [2]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=4
)
state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model = model.to(DEVICE)
model.eval()
print("model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model loaded


In [3]:
def extract_cls_embeddings(texts, batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch,
            max_length=128,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )
        input_ids      = encoded["input_ids"].to(DEVICE)
        attention_mask = encoded["attention_mask"].to(DEVICE)
        with torch.no_grad():
            outputs = model.distilbert(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            cls = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(cls.cpu().numpy())
    return np.concatenate(all_embeddings, axis=0)

print("extractor defined")

extractor defined


In [4]:
SAMPLES_PER_CLASS_REF = 148
SAMPLES_PER_CLASS_DAY = 64
N_CLASSES = 4
SIM_DAYS  = 60

AG_NEWS_INJECTION_DAYS = {
    3, 6, 9, 13, 16, 19, 22, 25, 28, 31,
    34, 37, 40, 43, 46, 49, 51, 54, 57, 59
}

label_groups = {
    c: df[df["label"] == c].reset_index(drop=True)
    for c in range(N_CLASSES)
}

ref_df = pd.concat([
    label_groups[c].iloc[:SAMPLES_PER_CLASS_REF]
    for c in range(N_CLASSES)
]).reset_index(drop=True)

print(f"reference window size: {len(ref_df)}")
print(f"label distribution:\n{ref_df['label'].value_counts().sort_index()}")

reference window size: 592
label distribution:
label
0    148
1    148
2    148
3    148
Name: count, dtype: int64


In [5]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

ref_embeddings = extract_cls_embeddings(ref_df["text"].tolist())
np.save(f"{OUTPUT_DIR}/ref_embeddings.npy", ref_embeddings)

print(f"ref_embeddings shape: {ref_embeddings.shape}")
print(f"expected: ({SAMPLES_PER_CLASS_REF * N_CLASSES}, 768)")

ref_embeddings shape: (592, 768)
expected: (592, 768)


In [6]:
for day in range(SIM_DAYS):
    day_df = pd.concat([
        label_groups[c].iloc[
            SAMPLES_PER_CLASS_REF + day * SAMPLES_PER_CLASS_DAY :
            SAMPLES_PER_CLASS_REF + (day + 1) * SAMPLES_PER_CLASS_DAY
        ]
        for c in range(N_CLASSES)
    ]).reset_index(drop=True)

    emb = extract_cls_embeddings(day_df["text"].tolist())
    np.save(f"{OUTPUT_DIR}/day_{day:02d}_embeddings.npy", emb)

    if (day + 1) % 10 == 0:
        print(f"day {day+1:02d}/60 done — shape: {emb.shape}")

print("all 60 clean days extracted")

day 10/60 done — shape: (256, 768)
day 20/60 done — shape: (256, 768)
day 30/60 done — shape: (256, 768)
day 40/60 done — shape: (256, 768)
day 50/60 done — shape: (256, 768)
day 60/60 done — shape: (256, 768)
all 60 clean days extracted


In [16]:
SWAP_RATIO   = 0.80
n_swap       = int(SWAP_RATIO * SAMPLES_PER_CLASS_DAY * N_CLASSES)
DOMINANT_CLASS = 1

for day in sorted(AG_NEWS_INJECTION_DAYS):
    day_start = SAMPLES_PER_CLASS_REF + day * SAMPLES_PER_CLASS_DAY

    all_clean = pd.concat([
        label_groups[c].iloc[day_start: day_start + SAMPLES_PER_CLASS_DAY]
        for c in range(N_CLASSES)
    ]).reset_index(drop=True)

    swap_source = label_groups[DOMINANT_CLASS].iloc[
        day_start + SAMPLES_PER_CLASS_DAY: day_start + SAMPLES_PER_CLASS_DAY + n_swap
    ]

    n_keep      = len(all_clean) - n_swap
    keep_rows   = all_clean.iloc[:n_keep]
    injected_df = pd.concat([swap_source, keep_rows]).reset_index(drop=True)

    emb = extract_cls_embeddings(injected_df["text"].tolist())
    np.save(f"{OUTPUT_DIR}/day_{day:02d}_embeddings_injected.npy", emb)
    print(f"day {day:02d} injected — shape: {emb.shape}  "
          f"dominant_class_pct: {len(swap_source)/len(injected_df)*100:.0f}%")

print("all 20 injection days re-extracted with dominant class injection")

day 03 injected — shape: (256, 768)  dominant_class_pct: 80%
day 06 injected — shape: (256, 768)  dominant_class_pct: 80%
day 09 injected — shape: (256, 768)  dominant_class_pct: 80%
day 13 injected — shape: (256, 768)  dominant_class_pct: 80%
day 16 injected — shape: (256, 768)  dominant_class_pct: 80%
day 19 injected — shape: (256, 768)  dominant_class_pct: 80%
day 22 injected — shape: (256, 768)  dominant_class_pct: 80%
day 25 injected — shape: (256, 768)  dominant_class_pct: 80%
day 28 injected — shape: (256, 768)  dominant_class_pct: 80%
day 31 injected — shape: (256, 768)  dominant_class_pct: 80%
day 34 injected — shape: (256, 768)  dominant_class_pct: 80%
day 37 injected — shape: (256, 768)  dominant_class_pct: 80%
day 40 injected — shape: (256, 768)  dominant_class_pct: 80%
day 43 injected — shape: (256, 768)  dominant_class_pct: 80%
day 46 injected — shape: (256, 768)  dominant_class_pct: 80%
day 49 injected — shape: (256, 768)  dominant_class_pct: 80%
day 51 injected — shape:

In [17]:
from sklearn.decomposition import PCA
import pickle

pca = PCA(n_components=50, random_state=42)
ref_pca = pca.fit_transform(ref_embeddings)

explained = pca.explained_variance_ratio_.sum()
print(f"PCA-50 explained variance: {explained:.4f}")

with open(f"{OUTPUT_DIR}/pca_model.pkl", "wb") as f:
    pickle.dump(pca, f)

print("pca_model.pkl saved")

PCA-50 explained variance: 0.9474
pca_model.pkl saved


In [18]:
clean_emb    = np.load(f"{OUTPUT_DIR}/day_03_embeddings.npy")
injected_emb = np.load(f"{OUTPUT_DIR}/day_03_embeddings_injected.npy")

detector2 = EmbeddingDriftDetector(rng_seed=42)
detector2.fit(ref_embeddings)

clean_result    = detector2.score(clean_emb)
injected_result = detector2.score(injected_emb)

print(f"clean day 03:")
print(f"  drift_score:    {clean_result['drift_score']}")
print(f"  drift_detected: {clean_result['drift_detected']}  (expect False)")
print(f"  mmd_threshold:  {clean_result['mmd_threshold']}")
print()
print(f"injected day 03:")
print(f"  drift_score:    {injected_result['drift_score']}")
print(f"  drift_detected: {injected_result['drift_detected']}  (expect True)")
print(f"  mmd_threshold:  {injected_result['mmd_threshold']}")

clean day 03:
  drift_score:    0.081445
  drift_detected: False  (expect False)
  mmd_threshold:  0.107512

injected day 03:
  drift_score:    0.439681
  drift_detected: True  (expect True)
  mmd_threshold:  0.107512


In [9]:
files = sorted(os.listdir(OUTPUT_DIR))
clean_files    = [f for f in files if f.startswith("day_") and not f.endswith("_injected.npy")]
injected_files = [f for f in files if f.endswith("_injected.npy")]

print(f"clean day files:    {len(clean_files)}   (expected 60)")
print(f"injected day files: {len(injected_files)}  (expected 20)")
print(f"total files:        {len(files)}           (expected 82)")
print(f"\nref_embeddings.npy present: {'ref_embeddings.npy' in files}")
print(f"pca_model.pkl present:      {'pca_model.pkl' in files}")

clean day files:    60   (expected 60)
injected day files: 20  (expected 20)
total files:        82           (expected 82)

ref_embeddings.npy present: True
pca_model.pkl present:      True


In [10]:
from sklearn.decomposition import PCA
from scipy import stats

clean_emb    = np.load(f"{OUTPUT_DIR}/day_03_embeddings.npy")
injected_emb = np.load(f"{OUTPUT_DIR}/day_03_embeddings_injected.npy")

clean_pca    = pca.transform(clean_emb)
injected_pca = pca.transform(injected_emb)
ref_pca_check = pca.transform(ref_embeddings)

def ks_check(ref, current, n_required=7):
    n_fail = 0
    for i in range(ref.shape[1]):
        _, p = stats.ks_2samp(ref[:, i], current[:, i])
        if p < 0.05:
            n_fail += 1
    return n_fail, n_fail >= n_required

n_clean, clean_drift       = ks_check(ref_pca_check, clean_pca)
n_injected, inject_drift   = ks_check(ref_pca_check, injected_pca)

print(f"clean day 03    — n_sig: {n_clean}/50    drift: {clean_drift}    (expect False)")
print(f"injected day 03 — n_sig: {n_injected}/50    drift: {inject_drift}    (expect True)")

clean day 03    — n_sig: 20/50    drift: True    (expect False)
injected day 03 — n_sig: 20/50    drift: True    (expect True)


In [14]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/greenmlops/src/features")
from embedding_drift import EmbeddingDriftDetector

detector = EmbeddingDriftDetector(rng_seed=42)
detector.fit(ref_embeddings)

clean_result    = detector.score(clean_emb)
injected_result = detector.score(injected_emb)

print(f"clean day 03:")
print(f"  drift_score:    {clean_result['drift_score']}")
print(f"  drift_detected: {clean_result['drift_detected']}  (expect False)")
print(f"  mmd_threshold:  {clean_result['mmd_threshold']}")
print()
print(f"injected day 03:")
print(f"  drift_score:    {injected_result['drift_score']}")
print(f"  drift_detected: {injected_result['drift_detected']}  (expect True)")
print(f"  mmd_threshold:  {injected_result['mmd_threshold']}")

clean day 03:
  drift_score:    0.081445
  drift_detected: False  (expect False)
  mmd_threshold:  0.107512

injected day 03:
  drift_score:    0.036238
  drift_detected: False  (expect True)
  mmd_threshold:  0.107512


In [15]:
clean_day_df = pd.concat([
    label_groups[c].iloc[
        SAMPLES_PER_CLASS_REF + 3 * SAMPLES_PER_CLASS_DAY :
        SAMPLES_PER_CLASS_REF + 4 * SAMPLES_PER_CLASS_DAY
    ]
    for c in range(N_CLASSES)
]).reset_index(drop=True)

print(f"clean day 03 label distribution:")
print(clean_day_df["label"].value_counts().sort_index())
print()

injected_texts_check = []
injected_labels_check = []
for c in range(N_CLASSES):
    day_start  = SAMPLES_PER_CLASS_REF + 3 * SAMPLES_PER_CLASS_DAY
    clean_rows = label_groups[c].iloc[day_start: day_start + SAMPLES_PER_CLASS_DAY]
    swap_class = ROTATION[c]
    swap_rows  = label_groups[swap_class].iloc[day_start: day_start + n_swap]

    window_texts  = swap_rows["text"].tolist() + clean_rows["text"].tolist()[n_swap:]
    window_labels = [swap_class] * n_swap + [c] * (SAMPLES_PER_CLASS_DAY - n_swap)
    injected_texts_check.extend(window_texts)
    injected_labels_check.extend(window_labels)

injected_label_series = pd.Series(injected_labels_check)
print(f"injected day 03 label distribution:")
print(injected_label_series.value_counts().sort_index())
print()
print(f"total injected samples: {len(injected_texts_check)}")

clean day 03 label distribution:
label
0    64
1    64
2    64
3    64
Name: count, dtype: int64

injected day 03 label distribution:
0    64
1    64
2    64
3    64
Name: count, dtype: int64

total injected samples: 256


In [19]:
print(f"{'day':<6} {'clean_score':<14} {'injected_score':<16} {'clean_ok':<10} {'inject_ok'}")
print("-" * 55)

all_clean_ok  = True
all_inject_ok = True

for day in sorted(AG_NEWS_INJECTION_DAYS):
    clean_emb_d    = np.load(f"{OUTPUT_DIR}/day_{day:02d}_embeddings.npy")
    injected_emb_d = np.load(f"{OUTPUT_DIR}/day_{day:02d}_embeddings_injected.npy")

    detector_d = EmbeddingDriftDetector(rng_seed=42)
    detector_d.fit(ref_embeddings)

    clean_r    = detector_d.score(clean_emb_d)
    injected_r = detector_d.score(injected_emb_d)

    c_ok = not clean_r["drift_detected"]
    i_ok = injected_r["drift_detected"]

    all_clean_ok  = all_clean_ok and c_ok
    all_inject_ok = all_inject_ok and i_ok

    print(f"{day:<6} {clean_r['drift_score']:<14} {injected_r['drift_score']:<16} "
          f"{str(c_ok):<10} {i_ok}")

print()
print(f"all clean days below threshold: {all_clean_ok}")
print(f"all injection days detected:    {all_inject_ok}")

day    clean_score    injected_score   clean_ok   inject_ok
-------------------------------------------------------
3      0.081445       0.439681         True       True
6      0.048446       0.44849          True       True
9      0.069306       0.443609         True       True
13     0.057204       0.448061         True       True
16     0.052044       0.448129         True       True
19     0.076236       0.449977         True       True
22     0.047336       0.451521         True       True
25     0.058973       0.441624         True       True
28     0.049686       0.44138          True       True
31     0.07872        0.449222         True       True
34     0.065392       0.440041         True       True
37     0.058721       0.440937         True       True
40     0.076116       0.447092         True       True
43     0.08842        0.440063         True       True
46     0.08328        0.452572         True       True
49     0.058799       0.441509         True       True
51  